In [ ]:
# Key Management

import os
os.environ['AWS_BEARER_Token_BEDROCK'] = ""

In [2]:
# Client Setup
import boto3

client = boto3.client("bedrock-runtime", region_name="us-east-2")
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"

In [3]:
# Helper functions


def add_user_message(messages, content):
    if isinstance(content, str):
        user_message = {"role": "user", "content": [{"text": content}]}
    else:
        user_message = {"role": "user", "content": content}
    messages.append(user_message)


def add_assistant_message(messages, content):
    if isinstance(content, str):
        assistant_message = {
            "role": "assistant",
            "content": [{"text": content}],
        }
    else:
        assistant_message = {"role": "assistant", "content": content}

    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    tool_choice="auto",
    text_editor=None,
    thinking=False,
    thinking_budget=1024,
):
    
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "stopSequences": stop_sequences,
        },
    }

    if system:
        params["system"] = [{"text": system}]

    tool_choices = {
        "auto": {"auto": {}},
        "any": {"any": {}},
    }
    if tools or text_editor:
        choice = tool_choices.get(tool_choice, {"tool": {"name": tool_choice}})
        params["toolConfig"] = {"tools": tools, "toolChoice": choice}


    additional_model_fields = {}
    if text_editor:
        additional_model_fields["tools"] = [
            {
                "type": text_editor,
                "name": "str_replace_editor",
            }
        ]
        
    if thinking:
        additional_model_fields["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget, 
        }

    params["additionalModelRequestFields"] = additional_model_fields
    response = client.converse(**params)
    parts = response["output"]["message"]["content"]

    return {
        "parts": parts,
        "stop_reason": response["stopReason"],
        "text": "\n".join([p["text"] for p in parts if "text" in p]),
    }

In [4]:
messages = []

add_user_message(
    messages,
    "Write a one paragraph guide to recursion",
)

result = chat(messages, thinking=True)

result["parts"]

[{'reasoningContent': {'reasoningText': {'text': "I need to write a clear, concise paragraph explaining recursion. Recursion is when a function or process calls itself as part of its execution, solving complex problems by breaking them down into simpler instances of the same problem. I should explain this concept, mention why it's useful, include an example, and touch on how it works fundamentally with base cases and recursive cases. I'll aim to keep it informative yet accessible for someone unfamiliar with the concept.",
    'signature': 'ErcBCkgICxABGAIiQK3ECDIaB5zoR6bjxSC76D/wqlCBubAG7mfLFrV/cLiOJ2tIFESaVgEW2+Zl3lNYMgm3FeXjX8SrJrvTTK7aOPoSDCKmvZIq9cok2LeKSRoM8DqIRy2ESqRkabcQIjAAvgoRcKYMP/6q/vYI3TGE4EEfL/5vTYpnFyn0B9uObsWmVDw8dMRnw88Y6e5gKPwqHUrhkS8bZAi2OJT0pM6jXVU9wenFyWwoLWWQZmekGAI='}}},
 {'text': '# A Brief Guide to Recursion\n\nRecursion is a powerful programming technique in which a function calls itself to solve a problem by breaking it down into smaller, similar subproblems. 